# Load Removed by DOI

In [5]:
import os

print(os.getcwd())

/home/liane/Documents/SEDS/Master thesis/ma_project/notebooks/paper_vis


In [7]:
import json
import pandas as pd


# import all papers excluded by langdetect
removed_doi = []
with open("../../data/filtered_out/no_doi.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        removed_doi.append(json.loads(line))

removed_df = pd.DataFrame(removed_doi)
removed_df.shape

(60126, 11)

In [8]:
removed_df.columns

Index(['paperId', 'title', 'venue', 'year', 'citationCount', 'fieldsOfStudy',
       'publicationTypes', 'authors', 'abstract', 'doi', 'pdf_url'],
      dtype='str')

In [10]:
removed_df["venue"].value_counts().head(20)

venue
                                                             57700
arXiv.org                                                      221
Annual Meeting of the Cognitive Science Society                 67
Text2Story@ECIR                                                 65
Alternative Therapies in Health and Medicine                    64
American Medical Informatics Association Annual Symposium       55
ICCC                                                            40
Tijdschrift voor psychiatrie                                    37
Psychiatria Danubina                                            36
Digital Humanities Conference                                   34
Pain Physician                                                  34
FNP                                                             32
Georgian medical news                                           28
Journal of the Canadian Chiropractic Association                23
SemEval@ACL                                             

# whole dataset

In [11]:
# import all semscho papers before any filtering
import pandas as pd
import glob
import json

# ---------------------------------------------------
# 1. Find all batch files
# ---------------------------------------------------

files = glob.glob("../../data/raw/search_results_batch_*.jsonl")

print(f"Found {len(files)} files")

# ---------------------------------------------------
# 2. Load all records
# ---------------------------------------------------

records = []

for file in files:
    with open(file, "r", encoding="utf-8") as f:
        for line in f:
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                print(f"Skipping malformed line in {file}")

# ---------------------------------------------------
# 3. Convert to DataFrame
# ---------------------------------------------------

df = pd.DataFrame(records)

print(df.shape)
print(df.columns)

Found 6 files
(547910, 12)
Index(['paperId', 'externalIds', 'url', 'title', 'venue', 'year',
       'citationCount', 'openAccessPdf', 'fieldsOfStudy', 'publicationTypes',
       'authors', 'abstract'],
      dtype='str')


In [12]:
# is paperId not unique??
# no it is not and I already solved this problem earlier logs in data/processed/filter_log.jsonl - dups removed: 22146
print(len(df["paperId"].unique()))
len(df) - len(df["paperId"].unique())

525764


22146

In [13]:
# remove dups
df = df.drop_duplicates(subset="paperId", keep="first")
df.shape

(525764, 12)

In [16]:
#remove
with open("../../data/filtered_out/language_langdetect.jsonl", "r", encoding="utf-8") as f:
    removed_langdetect = [json.loads(line)["paperId"] for line in f]

len(removed_langdetect)

df = df[~df["paperId"].isin(removed_langdetect)]

# combine non-doi with total

In [48]:
# Count venues in removed_df
removed_counts = removed_df["venue"].value_counts().rename("removed_count")

# Count venues in whole_df
whole_counts = df["venue"].value_counts().rename("whole_count")

# Combine them
venue_comparison = pd.concat([removed_counts, whole_counts], axis=1).fillna(0)

# Make counts integers
venue_comparison = venue_comparison.astype(int)

# Show percentage
venue_comparison["percentage_removed"] = venue_comparison["removed_count"]/ venue_comparison["whole_count"]

#venue_comparison["percentage_removed"] = (
venue_comparison["removed_perc"] = (venue_comparison["removed_count"] / venue_comparison["whole_count"] * 100).round(2).astype(str) + "%"

# remove all venues that have a removed_count of 0
venue_comparison = venue_comparison[venue_comparison["removed_count"]>0]

venue_comparison[["removed_count", "removed_perc"]].head(21)

,removed_count,removed_perc
venue,,
,57700,37.23%
arXiv.org,221,16.31%
Annual Meeting of the Cognitive Science Society,67,91.78%
Text2Story@ECIR,65,91.55%
Alternative Therapies in Health and Medicine,64,100.0%
American Medical Informatics Association Annual Symposium,55,96.49%
ICCC,40,95.24%
Tijdschrift voor psychiatrie,37,100.0%
Psychiatria Danubina,36,63.16%


In [26]:
venue_comparison[venue_comparison["removed_count"] == venue_comparison["whole_count"]].value_counts(removed_counts)

venue_comparison[venue_comparison["removed_count"] == venue_comparison["whole_count"]].head(20)

,removed_count,whole_count,percentage_removed
venue,,,
Alternative Therapies in Health and Medicine,64,64,1.0
Tijdschrift voor psychiatrie,37,37,1.0
FNP,32,32,1.0
Georgian medical news,28,28,1.0
Journal of the Canadian Chiropractic Association,23,23,1.0
Conference and Labs of the Evaluation Forum,23,23,1.0
Canadian family physician Medecin de famille canadien,18,18,1.0
Americas Conference on Information Systems,15,15,1.0
Canadian journal of dental hygiene,14,14,1.0


# OTHER STUFF (TO BE REMOVED)

In [14]:
# percentage removed:
len(removed_df)/len(df)*100

11.435929428412749

In [10]:
# add non-english marker to the rows removed
df["removed"] = df["paperId"].isin(removed_df["paperId"])
df["removed"].value_counts()

removed
False    491531
True      34233
Name: count, dtype: int64

# OA metadata language removal

In [46]:
with open("../data/filtered_out/language_langdetect.jsonl", "r", encoding="utf-8") as f:
    removed_langdetect = [json.loads(line)["paperId"] for line in f]

len(removed_langdetect)

34233

In [47]:
with open("../data/filtered_out/ngram_exclusion.jsonl", "r", encoding="utf-8") as f:
    removed_ngram = [json.loads(line)["paperId"] for line in f]

len(removed_ngram)

85522

In [48]:
with open("../data/filtered_out/no_doi.jsonl", "r", encoding="utf-8") as f:
    removed_doi = [json.loads(line)["paperId"] for line in f]

len(removed_doi)

60126

In [49]:
with open("../data/filtered_out/no_match_oa.jsonl", "r", encoding="utf-8") as f:
    removed_noMatchOA = [json.loads(line)["paperId"] for line in f]

len(removed_noMatchOA)

2342

In [50]:
with open("../data/filtered_out/language_oa.jsonl", "r", encoding="utf-8") as f:
    removed_oa = [json.loads(line)["paperId"] for line in f]

len(removed_oa)

28539

In [55]:
import numpy as np

other_ids = (
    set(removed_langdetect)
    | set(removed_ngram)
    | set(removed_doi)
    | set(removed_noMatchOA)
)

df["removal_category"] = np.select(
    [
        df["paperId"].isin(other_ids),
        df["paperId"].isin(removed_oa),
    ],
    [
        "other",
        "OpenAlex",
    ],
    default="not removed"
)